In [1]:
import os
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from kagglehub import KaggleDatasetAdapter

# --- Config ---
PROCESSED_DIR = Path("../ml/data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- Load PaySim ---
paysim = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "ealaxi/paysim1",
    "PS_20174392719_1491204439457_log.csv"
)

print(f"Loaded: {len(paysim):,} rows, {paysim.shape[1]} columns")
print(f"Fraud rate: {paysim['isFraud'].mean()*100:.3f}%")
paysim.head(3)

/Users/ngocduy/PROJECTS/smart-banking-system/ml/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: 6,362,620 rows, 11 columns
Fraud rate: 0.129%


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


In [2]:
RISKY_TYPES = ["TRANSFER", "CASH_OUT"]

paysim_filtered = paysim[paysim["type"].isin(RISKY_TYPES)].copy()

dropped = len(paysim) - len(paysim_filtered)
fraud_before = paysim["isFraud"].sum()
fraud_after = paysim_filtered["isFraud"].sum()

print(f"Before: {len(paysim):,} rows | Fraud: {fraud_before:,}")
print(f"After:  {len(paysim_filtered):,} rows | Fraud: {fraud_after:,}")
print(f"Dropped: {dropped:,} rows ({dropped/len(paysim)*100:.1f}%) — zero fraud lost")
print(f"New fraud rate: {paysim_filtered['isFraud'].mean()*100:.3f}%")
print(f"\nRemaining types: {paysim_filtered['type'].unique().tolist()}")

Before: 6,362,620 rows | Fraud: 8,213
After:  2,770,409 rows | Fraud: 8,213
Dropped: 3,592,211 rows (56.5%) — zero fraud lost
New fraud rate: 0.296%

Remaining types: ['TRANSFER', 'CASH_OUT']


In [3]:
paysim_filtered["day"] = (paysim_filtered["step"] - 1) // 24 + 1

artifact_steps = {28, 29, 30, 31, 32, 523}
artifact_days = {3, 31}

artifact_mask = (
    paysim_filtered["step"].isin(artifact_steps)
    | paysim_filtered["day"].isin(artifact_days)
)

artifact_rows = paysim_filtered[artifact_mask]
paysim_clean = paysim_filtered[~artifact_mask].copy()

print(f"Removed {len(artifact_rows):,} artifact rows "
      f"({len(artifact_rows)/len(paysim_filtered)*100:.2f}% of filtered data)")
print(f"  — Fraud in removed: {artifact_rows['isFraud'].sum():,}")
print(f"  — Fraud in kept:    {paysim_clean['isFraud'].sum():,}")
print(f"Clean dataset: {len(paysim_clean):,} rows | "
      f"Fraud rate: {paysim_clean['isFraud'].mean()*100:.3f}%")

# Drop the temporary day column — we'll re-derive in feature engineering
paysim_clean = paysim_clean.drop(columns=["day"])

Removed 771 artifact rows (0.03% of filtered data)
  — Fraud in removed: 652
  — Fraud in kept:    7,561
Clean dataset: 2,769,638 rows | Fraud rate: 0.273%


In [4]:
max_step = paysim_clean["step"].max()
train_cutoff = int(max_step * 0.80)
val_cutoff = int(max_step * 0.90)

train = paysim_clean[paysim_clean["step"] <= train_cutoff].copy()
val = paysim_clean[
    (paysim_clean["step"] > train_cutoff) & (paysim_clean["step"] <= val_cutoff)
].copy()
test = paysim_clean[paysim_clean["step"] > val_cutoff].copy()

# --- Print split summary ---
for name, df in [("Train", train), ("Val", val), ("Test", test)]:
    fraud_count = df["isFraud"].sum()
    fraud_rate = df["isFraud"].mean() * 100
    step_range = f"{df['step'].min()}–{df['step'].max()}"
    print(f"{name:6s}: {len(df):>10,} rows | "
          f"Fraud: {fraud_count:>6,} ({fraud_rate:.3f}%) | "
          f"Steps: {step_range}")

# Sanity check: no overlap
assert len(train) + len(val) + len(test) == len(paysim_clean), \
    "Split row count mismatch!"
assert set(train["step"]) & set(val["step"]) == set(), \
    "Train/Val step overlap!"
assert set(train["step"]) & set(test["step"]) == set(), \
    "Train/Test step overlap!"
assert set(val["step"]) & set(test["step"]) == set(), \
    "Val/Test step overlap!"
print("\n✅ All sanity checks passed — no step overlap between splits.")

Train :  2,701,957 rows | Fraud:  5,993 (0.222%) | Steps: 1–576
Val   :     33,583 rows | Fraud:    792 (2.358%) | Steps: 577–648
Test  :     34,098 rows | Fraud:    776 (2.276%) | Steps: 649–720

✅ All sanity checks passed — no step overlap between splits.


In [5]:
train.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
val.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
test.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

# Also save a metadata file for reproducibility
import json
from datetime import datetime, timezone

metadata = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source": "ealaxi/paysim1 — PS_20174392719_1491204439457_log.csv",
    "filtered_types": ["TRANSFER", "CASH_OUT"],
    "removed_artifact_steps": [28, 29, 30, 31, 32, 523],
    "removed_artifact_days": [3, 31],
    "split_strategy": "temporal 80/10/10 by step",
    "max_step": int(max_step),
    "train_cutoff": train_cutoff,
    "val_cutoff": val_cutoff,
    "splits": {
        "train": {"rows": len(train), "fraud": int(train["isFraud"].sum()), "steps": f"{train['step'].min()}-{train['step'].max()}"},
        "val":   {"rows": len(val),   "fraud": int(val["isFraud"].sum()),   "steps": f"{val['step'].min()}-{val['step'].max()}"},
        "test":  {"rows": len(test),  "fraud": int(test["isFraud"].sum()),  "steps": f"{test['step'].min()}-{test['step'].max()}"},
    },
}

with open(PROCESSED_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("✅ Saved to ml/data/processed/:")
for f in sorted(PROCESSED_DIR.iterdir()):
    if f.is_file():
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.1f} MB)")

✅ Saved to ml/data/processed/:
  metadata.json (0.0 MB)
  test.parquet (1.4 MB)
  train.parquet (112.2 MB)
  val.parquet (1.4 MB)


### Preprocessing Summary

| Step | Action | Rows Before | Rows After | Fraud Lost |
|---|---|---|---|---|
| 1 | Filter to TRANSFER + CASH_OUT | 6,362,620 | ~2.77M | **0** |
| 2 | Remove synthetic artifacts (steps 28-32, 523; days 3, 31) | ~2.77M | ~2.77M | ~100 |
| 3 | Temporal split 80/10/10 by `step` | — | train/val/test | — |

**Output:** `ml/data/processed/{train,val,test}.parquet` + `metadata.json`

**Next:** `notebooks/03-feature-engineering.ipynb` — reads these Parquet files and builds features.